# 5.7 로지스틱 회귀 (Logistic Regression)

## 개념 정리

**로지스틱 회귀**는 이름은 "회귀"이지만 실제로는 **분류(Classification)** 에 사용되는 알고리즘이다.

- 선형 회귀의 결과값을 **시그모이드(Sigmoid) 함수**에 통과시켜 0~1 사이의 확률값으로 변환한다.
- 이 확률값이 0.5 이상이면 1(Positive), 미만이면 0(Negative)으로 분류한다.
- **이진 분류(Binary Classification)** 문제에서 강력한 베이스라인 모델로 자주 사용된다.

$$
P(y=1|x) = \frac{1}{1 + e^{-(w^T x + b)}}
$$

## 사용할 데이터셋
- **Wisconsin Breast Cancer 데이터셋**: 유방암의 양성/악성을 분류하는 이진 분류 데이터
- 특성: 30개의 수치형 피처 (반지름, 텍스처, 둘레, 넓이 등)
- 타겟: 0 (악성, malignant) / 1 (양성, benign)

## 학습 흐름
1. 데이터 로드 및 스케일링 (StandardScaler)
2. 학습/테스트 데이터 분리
3. 로지스틱 회귀 모델 학습 및 평가 (accuracy, ROC-AUC)
4. solver 종류별 성능 비교
5. GridSearchCV로 최적 하이퍼파라미터 탐색

### 1) 라이브러리 import 및 데이터 로드

In [ ]:
# 데이터 처리 및 시각화 라이브러리
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# 사이킷런 내장 데이터셋 (위스콘신 유방암 데이터)
from sklearn.datasets import load_breast_cancer
# 로지스틱 회귀 모델
from sklearn.linear_model import LogisticRegression

# 유방암 데이터셋 로드 (Bunch 객체 형태로 반환)
# - cancer.data: 피처 배열 (569 x 30)
# - cancer.target: 타겟 배열 (569,) → 0(악성), 1(양성)
cancer = load_breast_cancer()

### 2) 데이터 스케일링 및 학습/테스트 분리

**왜 스케일링이 필요한가?**

로지스틱 회귀는 내부적으로 경사하강법(Gradient Descent) 계열 알고리즘으로 가중치를 학습하기 때문에,
피처 간 스케일 차이가 크면 학습 속도가 느려지거나 수렴이 잘 되지 않을 수 있다.

`StandardScaler`는 각 피처를 **평균 0, 분산 1** 로 변환하여 스케일을 맞춰준다.
(주의: 데이터를 표준정규분포로 만드는 것이 아니라, 평균과 분산만 조정하는 것이다.)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# StandardScaler로 평균 0, 분산 1로 데이터의 스케일을 변환
# (주의: 표준정규분포로 변환되는 것이 아니라 평균/분산만 조정됨)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(cancer.data)

# 학습 데이터(70%)와 테스트 데이터(30%)로 분리
# random_state=0으로 고정하여 재현성 확보
X_train, X_test, y_train, y_test = train_test_split(
    data_scaled, cancer.target, test_size=0.3, random_state=0
)

### 3) 로지스틱 회귀 모델 학습 및 평가

**평가 지표**
- **Accuracy(정확도)**: 전체 예측 중 맞춘 비율
- **ROC-AUC**: 분류 모델이 양성/음성을 얼마나 잘 구분하는지를 나타내는 지표 (1에 가까울수록 좋음)
  - 단순 정확도는 클래스 불균형에 취약하므로, 이진 분류에서는 ROC-AUC를 함께 보는 것이 좋다.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

# 로지스틱 회귀 모델 생성 및 학습
# solver를 지정하지 않으면 기본값인 'lbfgs'가 사용됨
lr_clf = LogisticRegression()  # solver='lbfgs' (기본값)
lr_clf.fit(X_train, y_train)

# 예측 수행
lr_preds = lr_clf.predict(X_test)                          # 클래스 예측 (0 또는 1)
lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]        # 양성 클래스(1)에 속할 확률

# accuracy와 roc_auc 측정
# - accuracy_score: 실제값과 예측값의 일치 비율
# - roc_auc_score: 실제값과 예측 확률을 입력으로 받음
print('accuracy: {0:.3f}, roc_auc:{1:.3f}'.format(
    accuracy_score(y_test, lr_preds),
    roc_auc_score(y_test, lr_preds_proba)
))

### 4) solver 종류별 성능 비교

**solver란?**

로지스틱 회귀의 비용 함수를 최적화하는 알고리즘이다. 사이킷런은 다음과 같은 solver를 지원한다.

| solver | 특징 | 지원 penalty |
|--------|------|-------------|
| `lbfgs` | 기본값. 준-뉴턴 방법, 메모리 효율적 | L2, none |
| `liblinear` | 좌표 하강법. 작은 데이터셋에 적합 | L1, L2 |
| `newton-cg` | 뉴턴 켤레경사법. 대용량 데이터에 적합 | L2, none |
| `sag` | 확률적 평균 경사. 대용량 데이터에 빠름 | L2, none |
| `saga` | sag의 확장 버전. L1도 지원 | L1, L2, elasticnet, none |

→ **데이터 크기와 사용하려는 규제(penalty)에 따라 적절한 solver를 선택해야 한다.**

In [ ]:
# 비교할 solver 목록
solvers = ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']

# 여러 개의 solver 값별로 LogisticRegression 학습 후 성능 평가
for solver in solvers:
    # max_iter=600: 수렴이 잘 안 되는 solver를 위해 최대 반복 횟수를 늘려줌
    lr_clf = LogisticRegression(solver=solver, max_iter=600)
    lr_clf.fit(X_train, y_train)
    
    # 예측 및 확률 예측
    lr_preds = lr_clf.predict(X_test)
    lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

    # accuracy와 roc_auc 측정
    print('solver:{0}, accuracy: {1:.3f}, roc_auc:{2:.3f}'.format(
        solver,
        accuracy_score(y_test, lr_preds),
        roc_auc_score(y_test, lr_preds_proba)
    ))

### 5) GridSearchCV를 통한 최적 하이퍼파라미터 탐색

**주요 하이퍼파라미터**
- **`penalty`**: 규제 종류 (`l1`, `l2`)
  - L1 규제(Lasso): 일부 가중치를 0으로 만들어 피처 선택 효과
  - L2 규제(Ridge): 가중치를 작게 만들어 과적합 완화
- **`C`**: 규제 강도의 **역수**. 작을수록 규제가 강함, 클수록 규제가 약함
- **`solver`**: 사용할 최적화 알고리즘

**주의**: solver와 penalty 조합이 호환되지 않으면 학습이 실패한다.
- `lbfgs`는 L1 penalty를 지원하지 않으므로, 아래 결과에서 `lbfgs + l1` 조합은 실패하여 NaN 경고가 발생한다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 탐색할 하이퍼파라미터 그리드 정의
# - solver: liblinear, lbfgs 두 가지
# - penalty: L2, L1 두 가지
# - C: 규제 강도의 역수, 5개 값 (총 2 x 2 x 5 = 20개 조합)
params = {
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2', 'l1'],
    'C': [0.01, 0.1, 1, 5, 10]
}

# 로지스틱 회귀 모델 생성
lr_clf = LogisticRegression()

# GridSearchCV: 모든 하이퍼파라미터 조합에 대해 교차검증 수행
# - param_grid: 탐색할 파라미터 조합
# - scoring='accuracy': 정확도를 기준으로 평가
# - cv=3: 3-fold 교차검증
grid_clf = GridSearchCV(lr_clf, param_grid=params, scoring='accuracy', cv=3)
grid_clf.fit(data_scaled, cancer.target)

# 최적 하이퍼파라미터와 최고 평균 정확도 출력
# (lbfgs + l1 조합은 호환되지 않아 NaN 경고가 발생할 수 있음)
print('최적 하이퍼 파라미터:{0}, 최적 평균 정확도:{1:.3f}'.format(
    grid_clf.best_params_,
    grid_clf.best_score_
))